# Cricket Match Winner Prediction Using Logistic Regression

In this project, we will use real international T20 cricket match data from Cricsheet.

We will:
1. Download real cricket match data.
2. Display the original raw data.
3. Clean the data using Universal Data Cleaner.
4. Compare raw and cleaned data.
5. Train a Logistic Regression model.
6. Evaluate the model.

In [4]:
import pandas as pd
import numpy as np
import requests
import zipfile
import json
import io

from data_cleaner import DataCleaner, CleanerConfig

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

## Step 2: Downloading the Cricket Dataset

We are downloading real Men's T20 International match data from Cricsheet.
The dataset is provided in JSON format inside a ZIP archive.

In [5]:
url = "https://cricsheet.org/downloads/t20s_male_json.zip"

response = requests.get(url, timeout=180)

response.raise_for_status()

zip_data = zipfile.ZipFile(io.BytesIO(response.content))

json_files = [
    file for file in zip_data.namelist()
    if file.endswith(".json")
]

print("Dataset downloaded successfully!")
print("Total JSON files:", len(json_files))
print("First 5 files:", json_files[:5])

Dataset downloaded successfully!
Total JSON files: 3649
First 5 files: ['1496587.json', '1528302.json', '1529230.json', '1529792.json', '1552306.json']


## Step 3: Loading Raw Cricket Data

In this step, we will read the JSON files from the downloaded ZIP archive and convert the match information into a Pandas DataFrame.

We will display the original raw data before applying any cleaning.

In [6]:
matches = []

for file in json_files:
    with zip_data.open(file) as json_file:
        match_data = json.load(json_file)

    info = match_data["info"]

    teams = info.get("teams", [])
    toss = info.get("toss", {})
    outcome = info.get("outcome", {})

    winner = outcome.get("winner")

    if len(teams) == 2 and winner in teams:
        match = {
            "team_1": teams[0],
            "team_2": teams[1],
            "venue": info.get("venue"),
            "date": info.get("dates", [None])[0],
            "toss_winner": toss.get("winner"),
            "toss_decision": toss.get("decision"),
            "winner": winner
        }

        matches.append(match)

raw_df = pd.DataFrame(matches)

print("Raw Dataset Shape:", raw_df.shape)

display(raw_df.head(10))

Raw Dataset Shape: (3530, 7)


,team_1,team_2,venue,date,toss_winner,toss_decision,winner
0,Sri Lanka,England,Old Trafford,2026-09-19,England,field,England
1,Portugal,France,Santarem Cricket Ground,2026-04-06,Portugal,bat,Portugal
2,West Indies,India,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,2026-10-06,India,field,India
3,Sweden,Indonesia,"Udayana Cricket Ground, Bali",2026-04-11,Sweden,bat,Sweden
4,Japan,Hong Kong,Sano International Cricket Ground,2026-09-19,Japan,bat,Hong Kong
5,Hong Kong,Malaysia,Korogi Sports Park,2026-09-24,Hong Kong,bat,Malaysia
6,Hong Kong,Oman,Korogi Sports Park,2026-09-25,Hong Kong,bat,Hong Kong
7,Bangladesh,Pakistan,Korogi Sports Park,2026-10-01,Pakistan,field,Pakistan
8,India,Sri Lanka,Korogi Sports Park,2026-10-01,Sri Lanka,field,India
9,Sri Lanka,Bangladesh,Korogi Sports Park,2026-10-03,Sri Lanka,bat,Sri Lanka


## Step 4: Cleaning the Raw Dataset

We will use our Universal Data Cleaner to handle missing values and improve data quality.

The original winner column will be kept separate because it is our prediction target.

In [7]:
X_original = raw_df.drop(columns=["winner"]).copy()

cleaner = DataCleaner(CleanerConfig())

cleaner.fit(X_original)

X_cleaned = cleaner.transform(X_original)

cleaned_df = X_cleaned.copy()
cleaned_df["winner"] = raw_df["winner"].values

print("Original Dataset Shape:", raw_df.shape)
print("Cleaned Dataset Shape:", cleaned_df.shape)

print("\nCleaned Dataset:")
display(cleaned_df.head(10))

Original Dataset Shape: (3530, 7)
Cleaned Dataset Shape: (3530, 7)

Cleaned Dataset:


,team_1,team_2,venue,date,toss_winner,toss_decision,winner
0,Sri Lanka,England,Old Trafford,2026-09-19,England,field,England
1,Portugal,France,Santarem Cricket Ground,2026-04-06,Portugal,bat,Portugal
2,West Indies,India,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,2026-10-06,India,field,India
3,Sweden,Indonesia,"Udayana Cricket Ground, Bali",2026-04-11,Sweden,bat,Sweden
4,Japan,Hong Kong,Sano International Cricket Ground,2026-09-19,Japan,bat,Hong Kong
5,Hong Kong,Malaysia,Korogi Sports Park,2026-09-24,Hong Kong,bat,Malaysia
6,Hong Kong,Oman,Korogi Sports Park,2026-09-25,Hong Kong,bat,Hong Kong
7,Bangladesh,Pakistan,Korogi Sports Park,2026-10-01,Pakistan,field,Pakistan
8,India,Sri Lanka,Korogi Sports Park,2026-10-01,Sri Lanka,field,India
9,Sri Lanka,Bangladesh,Korogi Sports Park,2026-10-03,Sri Lanka,bat,Sri Lanka


## Step 5: Before and After Data Comparison

In [8]:
print("========== RAW DATA ==========")
display(raw_df.head(10))

print("\n========== CLEANED DATA ==========")
display(cleaned_df.head(10))

comparison = pd.DataFrame({
    "Raw Missing Values": raw_df.isnull().sum(),
    "Cleaned Missing Values": cleaned_df.isnull().sum()
})

print("\n========== MISSING VALUES COMPARISON ==========")
display(comparison)

print("Raw Dataset Shape:", raw_df.shape)
print("Cleaned Dataset Shape:", cleaned_df.shape)

========== RAW DATA ==========


,team_1,team_2,venue,date,toss_winner,toss_decision,winner
0,Sri Lanka,England,Old Trafford,2026-09-19,England,field,England
1,Portugal,France,Santarem Cricket Ground,2026-04-06,Portugal,bat,Portugal
2,West Indies,India,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,2026-10-06,India,field,India
3,Sweden,Indonesia,"Udayana Cricket Ground, Bali",2026-04-11,Sweden,bat,Sweden
4,Japan,Hong Kong,Sano International Cricket Ground,2026-09-19,Japan,bat,Hong Kong
5,Hong Kong,Malaysia,Korogi Sports Park,2026-09-24,Hong Kong,bat,Malaysia
6,Hong Kong,Oman,Korogi Sports Park,2026-09-25,Hong Kong,bat,Hong Kong
7,Bangladesh,Pakistan,Korogi Sports Park,2026-10-01,Pakistan,field,Pakistan
8,India,Sri Lanka,Korogi Sports Park,2026-10-01,Sri Lanka,field,India
9,Sri Lanka,Bangladesh,Korogi Sports Park,2026-10-03,Sri Lanka,bat,Sri Lanka



========== CLEANED DATA ==========


,team_1,team_2,venue,date,toss_winner,toss_decision,winner
0,Sri Lanka,England,Old Trafford,2026-09-19,England,field,England
1,Portugal,France,Santarem Cricket Ground,2026-04-06,Portugal,bat,Portugal
2,West Indies,India,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,2026-10-06,India,field,India
3,Sweden,Indonesia,"Udayana Cricket Ground, Bali",2026-04-11,Sweden,bat,Sweden
4,Japan,Hong Kong,Sano International Cricket Ground,2026-09-19,Japan,bat,Hong Kong
5,Hong Kong,Malaysia,Korogi Sports Park,2026-09-24,Hong Kong,bat,Malaysia
6,Hong Kong,Oman,Korogi Sports Park,2026-09-25,Hong Kong,bat,Hong Kong
7,Bangladesh,Pakistan,Korogi Sports Park,2026-10-01,Pakistan,field,Pakistan
8,India,Sri Lanka,Korogi Sports Park,2026-10-01,Sri Lanka,field,India
9,Sri Lanka,Bangladesh,Korogi Sports Park,2026-10-03,Sri Lanka,bat,Sri Lanka



========== MISSING VALUES COMPARISON ==========


,Raw Missing Values,Cleaned Missing Values
team_1,0,0
team_2,0,0
venue,0,0
date,0,0
toss_winner,0,0
toss_decision,0,0
winner,0,0


Raw Dataset Shape: (3530, 7)
Cleaned Dataset Shape: (3530, 7)


## Step 6: Feature Engineering

We will prepare the input features and binary target variable for Logistic Regression.

Team 1 winning will be represented by 1, and Team 2 winning will be represented by 0.

In [9]:
model_df = cleaned_df.copy()

model_df["target"] = (
    model_df["winner"] == model_df["team_1"]
).astype(int)

model_df["toss_winner_is_team1"] = (
    model_df["toss_winner"] == model_df["team_1"]
).astype(int)

model_df["year"] = pd.to_datetime(
    model_df["date"]
).dt.year

X = model_df[
    [
        "team_1",
        "team_2",
        "venue",
        "toss_winner_is_team1",
        "toss_decision",
        "year"
    ]
]

y = model_df["target"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nTarget Distribution:")
display(y.value_counts())

Features Shape: (3530, 6)
Target Shape: (3530,)

Target Distribution:


target
0    1804
1    1726
Name: count, dtype: int64

## Step 7: Splitting the Dataset

We will divide the dataset into training and testing sets.

80% of the data will be used for training and 20% for testing.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

Training Features: (2824, 6)
Testing Features: (706, 6)
Training Target: (2824,)
Testing Target: (706,)


## Step 8: Encoding and Model Training

We will convert categorical features into numerical values using One-Hot Encoding.

Then we will train a Logistic Regression model using the training dataset.

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

categorical_features = [
    "team_1",
    "team_2",
    "venue",
    "toss_decision"
]

numerical_features = [
    "toss_winner_is_team1",
    "year"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numerical", StandardScaler(), numerical_features)
    ]
)

model = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

model.fit(X_train, y_train)

print("Logistic Regression Model Trained Successfully!")

Logistic Regression Model Trained Successfully!


## Step 9: Model Evaluation

We will evaluate our Logistic Regression model using:

1. Accuracy Score
2. ROC-AUC Score
3. Classification Report

In [12]:
y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_prob)

print("Model Accuracy:", round(accuracy * 100, 2), "%")
print("ROC-AUC Score:", round(roc_auc, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 65.01 %
ROC-AUC Score: 0.6996

Classification Report:
              precision    recall  f1-score   support

           0       0.65      0.68      0.66       361
           1       0.65      0.62      0.63       345

    accuracy                           0.65       706
   macro avg       0.65      0.65      0.65       706
weighted avg       0.65      0.65      0.65       706

